In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Assuming 'df' is your DataFrame with a column "lfp_py_norm_run" containing lists of power values
# Example: df = pd.DataFrame({"lfp_py_norm_run": [[0.1, 0.2, 0.3], [0.4, 0.5, 0.6]], "condition": ["A", "B"]})

# Step 1: Unpack the vectors into a long-form DataFrame
df = pd.read_pickle(r"S:\Sachuriga\Ephys_Recording\CR_CA1\LFP/LFp.pkl")

control_ids = ['65165', '65091', '63383', '66539', '65622']
exp_ids = ['65588', '63385', '66538', '66537', '66922']
data = []
for idx, row in df.iterrows():
    power_vector = row["lfp_py_norm_run"]  # The list of power values
    animal_id = row['animal_id']
    for power_lfp in power_vector:
        freqs = power_lfp.index.values   
        power_vector = power_lfp.values    # Frequency indices (0, 1, 2, ...)

        if animal_id in control_ids:
            condition="Control"
        elif animal_id in exp_ids:
            condition="Exp"

        for freq, power in zip(freqs, power_vector):
            data.append({
                "frequency": freq,             # Equivalent to 'timepoint'
                "power": power,                # Equivalent to 'signal'
                "condition": condition, # Equivalent to 'event' or 'region'
                "row_id": idx                  # Optional: track original row
            })

# Step 2: Convert to DataFrame
fmri_like_df = pd.DataFrame(data)
# Filter the DataFrame to include only frequencies from 0 to 100
fmri_like_df = fmri_like_df[(fmri_like_df['frequency'] >= 0) & (fmri_like_df['frequency'] <= 100)]

palette = {"Control": "blue", "Exp": "cyan"}
# Step 3: Plot using relplot
sns.relplot(
    data=fmri_like_df, 
    kind="line",
    x="frequency",          # Frequency on x-axis
    y="power",              # Power on y-axis
    col="condition",        # Separate plots by condition (like 'region')
    hue="condition",        # Color lines by condition (like 'event')
    style="condition",      # Line style by condition
    palette=palette
)

# Show the plot
plt.show()

In [ ]:
lfp_times = np.load(rf"{phy_folder}/{animals_id}/{phy_file}/lfp_times.npy")

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Assuming df is the input DataFrame with 'animal_id' and 'lfp_py_norm_run' columns
# Step 1: Get unique animal IDs
unique_animals = np.unique(df['animal_id'])
print(f"Number of unique animals: {len(unique_animals)}")
print(f"Animal IDs: {unique_animals}")

# Define control and experimental animal IDs
control_ids = ['65165', '65091', '63383', '66539', '65622']
exp_ids = ['65588', '63385', '66538', '66537', '66922']

# Define a common frequency grid (0 to 100 Hz, assuming 151 points for consistency)
common_frequencies = np.linspace(0, 400, 151)  # Adjust num_points if needed

# Step 2: Collect and average LFP data for each animal with index filtering
animal_lfp_averages = {}
frequency_indices = common_frequencies  # Use common frequencies for plotting

for animal_id in unique_animals:
    # Filter DataFrame for the current animal
    animal_data = df[df['animal_id'] == animal_id]["lfp_py_norm_run"]
    
    # Initialize a list to store filtered power values for this animal
    all_power_values = []
    
    # Iterate through each row's power vector
    for power_vector in animal_data:
        if isinstance(power_vector, list) and power_vector:
            # Assume power_vector[0] is a pandas Series with an index
            if isinstance(power_vector[0], pd.Series):
                power_series = power_vector[0]
                indices = power_series.index  # Use the Series index directly
                power_values = power_series.values  # Get the power values
                
                # Reindex or interpolate to common_frequencies
                if not np.array_equal(indices, common_frequencies):
                    # Interpolate to align with common_frequencies
                    interpolated_power = np.interp(
                        common_frequencies,
                        indices,
                        power_values,
                        left=np.nan,
                        right=np.nan
                    )
                else:
                    interpolated_power = power_values
                
                # Filter for indices where 0 <= index <= 100 (already ensured by common_frequencies)
                if len(interpolated_power) == len(common_frequencies):
                    all_power_values.append(interpolated_power)
            else:
                print(f"Warning: power_vector[0] for animal {animal_id} is not a pandas Series, skipping.")
    
    # Compute the average power for this animal
    if all_power_values:  # Check if there are any values
        try:
            all_power_values = np.vstack(all_power_values)
            average_power = np.nanmean(all_power_values, axis=0)  # Average across trials, ignoring NaNs
        except ValueError as e:
            print(f"Error stacking arrays for animal {animal_id}: {e}")
            average_power = np.nan
    else:
        average_power = np.nan  # Handle cases with no data
    
    # Store the result
    animal_lfp_averages[animal_id] = average_power

# Step 3: Create DataFrame with condition labels
average_lfp_df = pd.DataFrame({
    "animal_id": animal_lfp_averages.keys(),
    "average_lfp_power": animal_lfp_averages.values()
})

# Add conditionturbo column
average_lfp_df['condition'] = average_lfp_df['animal_id'].apply(
    lambda x: 'Control' if x in control_ids else 'Experimental' if x in exp_ids else 'Unknown'
)

# Filter out any animals not in control_ids or exp_ids
average_lfp_df = average_lfp_df[average_lfp_df['condition'] != 'Unknown']

# Step 4: Prepare data for plotting (convert to long format)
plot_data = []
for _, row in average_lfp_df.iterrows():
    animal_id = row['animal_id']
    condition = row['condition']
    power_vector = row['average_lfp_power']
    
    if isinstance(power_vector, np.ndarray) and not np.all(np.isnan(power_vector)):
        for freq_idx, power in zip(frequency_indices, power_vector):
            plot_data.append({
                'animal_id': animal_id,
                'condition': condition,
                'frequency': freq_idx,  # Use common_frequencies for x-axis
                'power': power
            })

plot_df = pd.DataFrame(plot_data)

# Step 5: Create relplot
sns.set(style="whitegrid")
g = sns.relplot(
    data=plot_df,
    x='frequency',
    y='power',
    hue='condition',
    style='condition',
    col='condition',
    kind='line',
    palette={'Control': 'blue', 'Experimental': 'cyan'},
    height=4,
    aspect=1.5,
    markers=True,
    dashes=False
)
# Customize plot
g.set_titles("{col_name}")
g.set_axis_labels("Frequency (Hz)", "Average LFP Power")
for ax in g.axes.flat:
    ax.set_yscale('log')  # Set y-axis to logarithmic scale
    ax.set_ylim(1e-4, 1e-0)  # Set y-axis limits from 10^-4 to 10^-1

# Customize plot
g.set_titles("{col_name}")
g.set_axis_labels("Frequency (Hz)", "Average LFP Power")
plt.suptitle("Average LFP Power by Condition (0 to 100 Hz)", y=1.05)
plt.show()

# Step 6: Display the DataFrame for reference
print("\nAverage LFP DataFrame with Conditions:")
print(average_lfp_df[['animal_id', 'condition']])

In [ ]:
control_ids = ['65165', '65091', '63383', '66539', '65622']
values=[]

control_lfp = []
for i  in range(len(df)):
    temp= df.iloc[i]
    temp_v = [] 
    for i1 in range(len(temp['lfp_py_norm_run'])):
        file_name=temp['session_id']
        delta_temp = np.sum(temp['lfp_py_norm_run'][i1].values[(temp['lfp_py_norm_run'][i1].index >= 0.5) & (temp['lfp_py_norm_run'][i1].index < 4)])
        theta_temp = np.sum(temp['lfp_py_norm_run'][i1].values[(temp['lfp_py_norm_run'][i1].index >= 4) & (temp['lfp_py_norm_run'][i1].index <= 12)])
        values.append(theta_temp/delta_temp)
        temp_v.append(theta_temp/delta_temp)
        if (theta_temp/delta_temp)<5:
            continue
        plt.figure(figsize=(12, 12))
        plt.plot(temp['lfp_py_norm_run'][i1][:100])
        plt.title(fr"{file_name}")
    if any(x < 5 for x in temp_v):
        continue
    control_lfp.append(df.iloc[[i]])

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Assuming 'df' is your DataFrame with a column "lfp_py_norm_run" containing lists of power values
# Example: df = pd.DataFrame({"lfp_py_norm_run": [[0.1, 0.2, 0.3], [0.4, 0.5, 0.6]], "condition": ["A", "B"]})

# Step 1: Unpack the vectors into a long-form DataFrame
df = df_compare
control_ids = ['65165', '65091', '63383', '66539', '65622']
exp_ids = ['65588', '63385', '66538', '66537', '66922']
data = []
for idx, row in df.iterrows():
    power_vector = row["lfp_py_norm_rest"]  # The list of power values
    animal_id = row['animal_id']
    for power_lfp in power_vector:
        freqs = power_lfp.index.values   
        power_vector = power_lfp.values    # Frequency indices (0, 1, 2, ...)

        if animal_id in control_ids:
            condition="Control"
        elif animal_id in exp_ids:
            condition="Exp"

        for freq, power in zip(freqs, power_vector):
            data.append({
                "frequency": freq,             # Equivalent to 'timepoint'
                "power": power,                # Equivalent to 'signal'
                "condition": condition, # Equivalent to 'event' or 'region'
                "row_id": idx                  # Optional: track original row
            })

# Step 2: Convert to DataFrame
fmri_like_df = pd.DataFrame(data)
# Filter the DataFrame to include only frequencies from 0 to 100
fmri_like_df = fmri_like_df[(fmri_like_df['frequency'] >= 0) & (fmri_like_df['frequency'] <= 100)]

palette = {"Control": "blue", "Exp": "cyan"}
# Step 3: Plot using relplot
sns.relplot(
    data=fmri_like_df, 
    kind="line",
    x="frequency",          # Frequency on x-axis
    y="power",              # Power on y-axis
    col="condition",        # Separate plots by condition (like 'region')
    hue="condition",        # Color lines by condition (like 'event')
    style="condition",      # Line style by condition
    palette=palette
)

# Show the plot
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Assuming 'df' is your DataFrame with a column "lfp_py_norm_run" containing lists of power values
# Example: df = pd.DataFrame({"lfp_py_norm_run": [[0.1, 0.2, 0.3], [0.4, 0.5, 0.6]], "condition": ["A", "B"]})

# Step 1: Unpack the vectors into a long-form DataFrame
df = df_compare
#df = pd.read_pickle(r"S:\Sachuriga\Ephys_Recording\CR_CA1\LFP/LFp.pkl")

control_ids = ['65165', '65091', '63383', '66539', '65622']
exp_ids = ['65588', '63385', '66538', '66537', '66922']
data = []
for idx, row in df.iterrows():
    power_vector = row["lfp_sr_norm_run"]  # The list of power values
    animal_id = row['animal_id']
    for power_lfp in power_vector:
        freqs = power_lfp.index.values   
        power_vector = power_lfp.values    # Frequency indices (0, 1, 2, ...)

        if animal_id in control_ids:
            condition="Control"
        elif animal_id in exp_ids:
            condition="Exp"

        for freq, power in zip(freqs, power_vector):
            data.append({
                "frequency": freq,             # Equivalent to 'timepoint'
                "power": power,                # Equivalent to 'signal'
                "condition": condition, # Equivalent to 'event' or 'region'
                "row_id": idx                  # Optional: track original row
            })

# Step 2: Convert to DataFrame
fmri_like_df = pd.DataFrame(data)
# Filter the DataFrame to include only frequencies from 0 to 100
fmri_like_df = fmri_like_df[(fmri_like_df['frequency'] >= 0) & (fmri_like_df['frequency'] <= 100)]

palette = {"Control": "blue", "Exp": "cyan"}
# Step 3: Plot using relplot
sns.relplot(
    data=fmri_like_df, 
    kind="line",
    x="frequency",          # Frequency on x-axis
    y="power",              # Power on y-axis
    col="condition",        # Separate plots by condition (like 'region')
    hue="condition",        # Color lines by condition (like 'event')
    style="condition",      # Line style by condition
    palette=palette
)

# Show the plot
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Assuming 'df' is your DataFrame with a column "lfp_py_norm_run" containing lists of power values
# Example: df = pd.DataFrame({"lfp_py_norm_run": [[0.1, 0.2, 0.3], [0.4, 0.5, 0.6]], "condition": ["A", "B"]})

# Step 1: Unpack the vectors into a long-form DataFrame

#df = pd.read_pickle(r"S:\Sachuriga\Ephys_Recording\CR_CA1\LFP/LFp.pkl")
df = df_compare
control_ids = ['65165', '65091', '63383', '66539', '65622']
exp_ids = ['65588', '63385', '66538', '66537', '66922']
data = []
for idx, row in df.iterrows():
    power_vector = row["lfp_sr_norm_rest"]  # The list of power values
    animal_id = row['animal_id']
    for power_lfp in power_vector:
        freqs = power_lfp.index.values   
        power_vector = power_lfp.values    # Frequency indices (0, 1, 2, ...)

        if animal_id in control_ids:
            condition="Control"
        elif animal_id in exp_ids:
            condition="Exp"

        for freq, power in zip(freqs, power_vector):
            data.append({
                "frequency": freq,             # Equivalent to 'timepoint'
                "power": power,                # Equivalent to 'signal'
                "condition": condition, # Equivalent to 'event' or 'region'
                "row_id": idx                  # Optional: track original row
            })

# Step 2: Convert to DataFrame
fmri_like_df = pd.DataFrame(data)
# Filter the DataFrame to include only frequencies from 0 to 100
fmri_like_df = fmri_like_df[(fmri_like_df['frequency'] >= 0) & (fmri_like_df['frequency'] <= 100)]

palette = {"Control": "blue", "Exp": "cyan"}
# Step 3: Plot using relplot
sns.relplot(
    data=fmri_like_df, 
    kind="line",
    x="frequency",          # Frequency on x-axis
    y="power",              # Power on y-axis
    col="condition",        # Separate plots by condition (like 'region')
    hue="condition",        # Color lines by condition (like 'event')
    style="condition",      # Line style by condition
    palette=palette
)

# Show the plot
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import shapiro, ttest_ind, mannwhitneyu

# Assuming df is your DataFrame
control_ids = ['65165', '65091', '63383', '66539', '65622']
exp_ids = ['65588', '63385', '66538', '66537', '66922']
variables = ['fast_event_rate_py', 'fast_theta_gamma_coupling_py', 'slow_event_rate_py', 'slow_theta_gamma_coupling_py',
             'fast_event_rate_sr', 'fast_theta_gamma_coupling_sr', 'slow_event_rate_sr', 'slow_theta_gamma_coupling_sr']

# Initialize an empty list to store results
data = []
unpacked_data=[]

for idx, row in df.iterrows():
    animal_id = row['animal_id']
    condition = "Control" if animal_id in control_ids else "Exp" if animal_id in exp_ids else None
    if condition:
        row_data = {"condition": condition, "row_id": idx}
        for var in variables:
            value = row[var]  # Assume scalar for simplicity
            row_data[var] = value
        data.append(row_data)

data_df = pd.DataFrame(data)
for _, row in data_df.iterrows():
    condition = row['condition']
    row_id = row['row_id']
    # Get the lists for each variable
    lists_per_var = {var: row[var] for var in variables}
    # Determine the length of the lists (assuming all lists in a row have the same length)
    list_length = len(lists_per_var[variables[0]]) if lists_per_var[variables[0]] else 0
    # Create a row for each index in the lists
    for i in range(list_length):
        new_row = {
            'condition': condition,
            'row_id': row_id,
            'list_index': i  # To track the position in the list
        }
        for var in variables:
            new_row[var] = lists_per_var[var][i] if i < len(lists_per_var[var]) else None
        unpacked_data.append(new_row)

# Create a new DataFrame from the unpacked data
unpacked_df = pd.DataFrame(unpacked_data)
control_color = 'blue'
exp_color = 'red'
# Create subplots
plt.figure(figsize=(15, 10))
for i, var in enumerate(variables, 1):
    # Create subplot
    ax = plt.subplot(2, 4, i)
    
    # # Boxplot with specified colors
    # sns.boxplot(x='condition', y=var, data=unpacked_df, 
    #             palette={'Control': 'blue', 'Exp': 'cyan'})
    
    sns.violinplot(
            data=unpacked_df, x='condition', y=var, 
            #ax=ax,
            inner = None,
            palette={"Control": control_color, "Exp": exp_color}, width=0.8, cut=0, linewidth=0
        )
    # Add individual points with matching colors
    sns.boxplot(
        data=unpacked_df, 
        x='condition', y=var, 
        palette={"Control": "black", "Exp": "black"},
        width=0.3, 
        fill=False,  # No fill, only outlines
        showfliers=False,  # Hide outliers
        showmeans=False,  # Remove mean marker, assuming midline is the median
        linewidth=1,  # Makes the lines narrower (thinner)
        #ax=ax  # Add this
    )
    #ax.set_ylabel(titles[idx])

    ax.set_xlabel('')
    ax.yaxis.grid(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(True)
    ax.spines['left'].set_visible(True)
    ax.set_xticklabels(['CR;DTA-', 'CR;DTA+'], rotation = -45)
    y_max = ax.get_ylim()[1]


    # Normality test
    control_data = unpacked_df[unpacked_df['condition'] == 'Control'][var].dropna()
    exp_data = unpacked_df[unpacked_df['condition'] == 'Exp'][var].dropna()
    
    # Shapiro test for normality
    stat_c, p_c = shapiro(control_data)
    stat_e, p_e = shapiro(exp_data)
    
    # Choose statistical test based on normality (p < 0.05 indicates non-normal)
    if p_c > 0.05 and p_e > 0.05:
        # Both normal: use t-test
        stat, p_val = ttest_ind(control_data, exp_data)
        test_name = 't-test'
    else:
        # At least one non-normal: use Mann-Whitney U
        stat, p_val = mannwhitneyu(control_data, exp_data)
        test_name = 'Mann-Whitney U'
    
    # Add title with statistical results
    plt.title(f'{var}\n{test_name}: p={p_val:.4f}', fontsize=10)
    
    # Adjust y-label
    plt.ylabel(var.split('_')[0] + '_' + var.split('_')[1])

# Adjust layout and display
plt.tight_layout()
plt.show()

# Print detailed statistical results
print("\nStatistical Analysis Results:")
print("-" * 50)
for var in variables:
    control_data = unpacked_df[unpacked_df['condition'] == 'Control'][var].dropna()
    exp_data = unpacked_df[unpacked_df['condition'] == 'Exp'][var].dropna()
    
    stat_c, p_c = shapiro(control_data)
    stat_e, p_e = shapiro(exp_data)
    
    if p_c > 0.05 and p_e > 0.05:
        stat, p_val = ttest_ind(control_data, exp_data)
        test_name = 't-test'
    else:
        stat, p_val = mannwhitneyu(control_data, exp_data)
        test_name = 'Mann-Whitney U'
    
    print(f"\n{var}:")
    print(f"Control normality (Shapiro): p={p_c:.4f}")
    print(f"Exp normality (Shapiro): p={p_e:.4f}")
    print(f"{test_name}: statistic={stat:.4f}, p-value={p_val:.4f}")

In [ ]:
unpacked_df

In [ ]:
data_df

In [ ]:
melted_df